# Basic Prompt Structures Tutorial

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/prompt-engineering/basic-prompt-structures.ipynb)

## Overview

This tutorial focuses on two fundamental types of prompt structures:
1. Single-turn prompts
2. Multi-turn prompts (conversations)

We'll use a local open-source model with HuggingFace Transformers to demonstrate these concepts.

## Motivation

Understanding different prompt structures is crucial for effective communication with AI models. Single-turn prompts are useful for quick, straightforward queries, while multi-turn prompts enable more complex, context-aware interactions. Mastering these structures allows for more versatile and effective use of AI in various applications.

## Key Components

1. **Single-turn Prompts**: One-shot interactions with the language model.
2. **Multi-turn Prompts**: Series of interactions that maintain context.
3. **Prompt Templates**: Reusable structures for consistent prompting using Python f-strings.
4. **Native Conversation History**: Maintaining context across multiple interactions using a simple message list.

## Method Details

We'll use HuggingFace Transformers with a locally-loaded open-source model (Qwen3-8B) to demonstrate these prompt structures. Conversation history is managed natively via a list of message dictionaries — no external frameworks needed. The tutorial will include practical examples and comparisons of different prompt types.

## Setup

First, let's install dependencies, load the model, and define our `generate()` helper.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

## Under the Hood: Chat Template Token Flow

Before we start prompting, let's understand what *actually happens* when you pass a list of
message dictionaries to a language model. The model doesn't see Python dicts — it sees a
flat sequence of **tokens** (integer IDs). The `apply_chat_template()` method is the bridge:

```
Python dicts  →  apply_chat_template()  →  formatted string  →  tokenizer.encode()  →  token IDs
```

Qwen3 uses the **ChatML** format. Each message is wrapped in special **control tokens**:
- `<|im_start|>` — marks the beginning of a message (followed by the role name)
- `<|im_end|>` — marks the end of a message

These control tokens are *not* regular words — they were added to the vocabulary specifically
to give the model structural awareness. During pre-training, the model learned that tokens
after `<|im_start|>system` define behavioral constraints, tokens after `<|im_start|>user`
are the human's request, and it should produce helpful content after `<|im_start|>assistant`.

Let's see this in action.

In [ ]:
# --- Under the Hood: inspect the exact token sequence the model sees ---

sample_messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": "The capital of France is Paris."},
]

# Step 1: apply_chat_template converts dicts → formatted string
raw_text = tokenizer.apply_chat_template(
    sample_messages, tokenize=False, add_generation_prompt=False, enable_thinking=False
)
print("=" * 70)
print("RAW TEMPLATE OUTPUT (what the model actually sees as text):")
print("=" * 70)
print(repr(raw_text))
print()

# Step 2: tokenize to get IDs
token_ids = tokenizer.encode(raw_text)
print(f"Total tokens: {len(token_ids)}")
print()

# Step 3: show every token — mark control tokens vs content tokens
# Qwen3 control tokens include <|im_start|>, <|im_end|>, <|endoftext|>
special_token_ids = set(tokenizer.all_special_ids)

print(f"{'IDX':>4}  {'TOKEN_ID':>8}  {'TYPE':<10}  DECODED")
print("-" * 70)
for idx, tid in enumerate(token_ids):
    decoded = tokenizer.decode([tid])
    token_type = "CONTROL" if tid in special_token_ids else "content"
    print(f"{idx:>4}  {tid:>8}  {token_type:<10}  {repr(decoded)}")

### What each control token does

| Token | Purpose |
|-------|--------|
| `<\|im_start\|>` | **Boundary marker**: tells the model "a new message begins here." The very next tokens specify the role (`system`, `user`, or `assistant`) followed by a newline. |
| `<\|im_end\|>` | **Termination marker**: tells the model "this message is complete." During generation, the model learns to *emit* this token when it's done responding. |

**Why does this matter?**

Without these delimiters, the model would have no way to distinguish where one speaker's
text ends and another's begins. The control tokens create **structural boundaries** in the
attention pattern — the model learns that information within one `<|im_start|>...<|im_end|>`
block is semantically cohesive. This is why using `apply_chat_template()` instead of
manually formatting strings is critical: the exact token IDs and placement must match
what the model saw during training.

## 1. Single-turn Prompts

Single-turn prompts are one-shot interactions with the language model. They consist of a single input (prompt) and generate a single output (response).

In [ ]:
single_turn_prompt = "What are the three primary colors?"
messages = [{"role": "user", "content": single_turn_prompt}]
print(generate(messages))

Now, let's use an f-string template to create a more structured single-turn prompt:

In [ ]:
topic = "color theory"
structured_prompt = f"Provide a brief explanation of {topic} and list its three main components."

messages = [{"role": "user", "content": structured_prompt}]
print(generate(messages))

### Why Single-Turn Works: Attention Focus

In a single-turn prompt, the model's **self-attention mechanism** has a clean, simple job.
Every attention head at every layer can dedicate 100% of its capacity to understanding the
relationship between tokens in *one instruction*.

Here's the intuition:

- The model produces the next token by computing a **probability distribution** over its
  entire vocabulary (~150K+ tokens for Qwen3).
- Each attention head looks at all previous tokens and decides which ones are *relevant*
  to predicting the next one.
- With a single, clear instruction, the attention weights concentrate on the key content
  words — making the output probability distribution **sharper** (higher confidence on the
  correct answer, lower probability on irrelevant tokens).

**Analogy**: Single-turn is like asking someone a question in a quiet room — they hear
exactly what you said, with no distracting context. The signal-to-noise ratio is maximal.

## 2. Multi-turn Prompts (Conversations)

Multi-turn prompts involve a series of interactions with the language model, allowing for more complex and context-aware conversations. We manage conversation history natively using a list of message dictionaries — each message has a `role` (system, user, or assistant) and `content`. The full history is passed to the model on every turn so it can maintain context.

In [ ]:
conversation_history = [
    {"role": "system", "content": "You are a helpful assistant."}
]

def chat(user_message, history=None):
    """Send a message and get a response, maintaining conversation history."""
    if history is None:
        history = conversation_history
    history.append({"role": "user", "content": user_message})
    response = generate(history)
    history.append({"role": "assistant", "content": response})
    return response

print(chat("Hi, I'm learning about space. Can you tell me about planets?"))
print(chat("What's the largest planet in our solar system?"))
print(chat("How does its size compare to Earth?"))

### Why Multi-Turn Works: Visible History and Attention Patterns

Multi-turn conversation works because of one fundamental property of transformer models:
**at every generation step, the model can attend to ALL previous tokens.**

When you pass `conversation_history` to `generate()`, the full history — system message,
every user turn, every assistant turn — is concatenated into one token sequence. The model
doesn't "remember" past turns in some hidden state; it literally *re-reads* everything.

This means:

1. **Coreference resolution works**: When you say "How does *its* size compare to Earth?",
   the model's attention heads can look back to the previous turn and find that "its" refers
   to Jupiter. The attention weight from "its" to "Jupiter" will be high.

2. **Context accumulates**: Each turn adds information. The system message sets behavioral
   priors, early turns establish topic, and later turns can refine or redirect.

3. **Coherence is "free"**: The model doesn't need special memory mechanisms — coherence
   emerges because all tokens are visible. This is why transformers replaced RNNs: no
   information bottleneck between turns.

**But there's a cost**: every turn you add increases the sequence length, and attention
computation is **O(n²)** in sequence length. We'll explore this tradeoff next.

Let's compare how single-turn and multi-turn prompts handle a series of related questions:

In [ ]:
# Single-turn prompts
prompts = [
    "What is the capital of France?",
    "What is its population?",
    "What is the city's most famous landmark?"
]

print("Single-turn responses:")
for prompt in prompts:
    messages = [{"role": "user", "content": prompt}]
    print(f"Q: {prompt}")
    print(f"A: {generate(messages)}\n")

# Multi-turn prompts
print("Multi-turn responses:")
multi_turn_history = [
    {"role": "system", "content": "You are a helpful assistant."}
]
for prompt in prompts:
    print(f"Q: {prompt}")
    print(f"A: {chat(prompt, history=multi_turn_history)}\n")

## Why Structure Matters: Context Window Deep Dive

Every language model has a **context window** — the maximum number of tokens it can process
at once. For Qwen3-8B, this is **128K tokens** (though practical limits are often lower due
to memory and speed).

Key insight: the context window is measured in **tokens, not characters**. Different content
tokenizes very differently:
- Common English words: ~1 token per word
- Technical jargon, code, rare words: 2-4+ tokens per word
- Whitespace and punctuation: often separate tokens

**Why this matters for prompt engineering**:

Self-attention complexity is **O(n²)** where n is the sequence length in tokens. This means:
- 2× more tokens → 4× more computation in attention layers
- 10× more tokens → 100× more computation

So every unnecessary token in your prompt literally costs quadratically more compute.
Let's see this in practice.

In [ ]:
# --- Context Window Deep Dive: token counting and growth ---

# Part A: Watch tokens grow turn by turn
print("=" * 70)
print("PART A: Token count growth across conversation turns")
print("=" * 70)

growing_history = [
    {"role": "system", "content": "You are a helpful astronomy tutor."},
]
turns = [
    ("user", "What is a star?"),
    ("assistant", "A star is a luminous sphere of plasma held together by gravity. Stars generate energy through nuclear fusion in their cores."),
    ("user", "How do stars form?"),
    ("assistant", "Stars form from molecular clouds — vast regions of gas and dust in space. When part of a cloud collapses under gravity, it heats up, and if the core reaches about 10 million Kelvin, nuclear fusion ignites and a star is born."),
    ("user", "What happens when a star runs out of fuel?"),
    ("assistant", "When a star exhausts its hydrogen fuel, its fate depends on its mass. Low-mass stars become white dwarfs, medium stars may become neutron stars, and the most massive stars collapse into black holes — often after a spectacular supernova explosion."),
]

for role, content in turns:
    growing_history.append({"role": role, "content": content})
    templated = tokenizer.apply_chat_template(
        growing_history, tokenize=False, add_generation_prompt=(role == "assistant"),
        enable_thinking=False
    )
    n_tokens = len(tokenizer.encode(templated))
    n_chars = len(content)
    ratio = n_tokens / max(n_chars, 1)
    print(f"  Turn {len(growing_history)-1:>2} ({role:>9}): "
          f"{n_tokens:>5} total tokens  |  "
          f"this msg: {n_chars:>4} chars → ~{ratio:.2f} tok/char")

print()

# Part B: When would we exceed the context window?
print("=" * 70)
print("PART B: Context window limits")
print("=" * 70)

context_window = 131072  # Qwen3's 128K context
avg_tokens_per_turn = n_tokens // (len(growing_history) - 1)  # rough average
max_turns_estimate = context_window // avg_tokens_per_turn
print(f"  Context window: {context_window:,} tokens")
print(f"  Average tokens/turn in our conversation: ~{avg_tokens_per_turn}")
print(f"  Estimated max turns before overflow: ~{max_turns_estimate}")
print(f"  (In practice, you'd hit memory/speed limits much sooner)")
print()

# Part C: Verbose vs concise — same meaning, different token cost
print("=" * 70)
print("PART C: Verbose vs concise — same semantics, different token cost")
print("=" * 70)

verbose_msg = [
    {"role": "user", "content": (
        "I would really appreciate it if you could kindly provide me with a "
        "comprehensive and detailed explanation of what photosynthesis is, "
        "including all of the various steps and processes that are involved "
        "in it, and also maybe describe why it is so important for life on Earth."
    )}
]

concise_msg = [
    {"role": "user", "content": "Explain photosynthesis: steps, processes, and importance for life on Earth."}
]

verbose_text = tokenizer.apply_chat_template(verbose_msg, tokenize=False, add_generation_prompt=True, enable_thinking=False)
concise_text = tokenizer.apply_chat_template(concise_msg, tokenize=False, add_generation_prompt=True, enable_thinking=False)

v_tokens = len(tokenizer.encode(verbose_text))
c_tokens = len(tokenizer.encode(concise_text))

print(f"  Verbose prompt: {v_tokens} tokens")
print(f"  Concise prompt: {c_tokens} tokens")
print(f"  Savings: {v_tokens - c_tokens} tokens ({100*(v_tokens-c_tokens)/v_tokens:.0f}% fewer)")
print(f"  Attention cost ratio (O(n²)): verbose needs ~{(v_tokens/c_tokens)**2:.1f}× more attention compute")

### Practical Implications

The numbers above reveal three critical prompt engineering principles:

1. **Token budget is finite**: Every conversation has a hard ceiling. Long system prompts,
   verbose instructions, and accumulated history all eat into the same budget. When you hit
   the limit, the model *cannot see* older content — it's as if those turns never happened.

2. **Conciseness saves more than you think**: Because attention is O(n²), a 2× reduction in
   tokens gives a 4× speedup in the attention layers. In production systems processing
   millions of requests, this translates directly to cost and latency.

3. **Attention dilution is real**: As the sequence gets longer, each attention head must
   spread its probability mass over more tokens. Important information from early in the
   conversation competes with recent tokens for attention — this is why models sometimes
   "forget" early instructions in long conversations. This phenomenon is related to the
   well-documented **lost in the middle** effect.

## Expanded Prompt Structure Patterns

Beyond basic single-turn and multi-turn, there are several proven prompt structures that
leverage the model's attention patterns in specific ways. Each pattern works because it
organizes information to make the model's job *easier* — reducing ambiguity and focusing
attention on what matters.

In [ ]:
# --- Pattern 1: Instruction + Context ---
# Why it works: Separating context from the question lets the model attend to
# the context as background and the question as the generation target.
# The model learns "material before the question = reference; after = what to produce."

print("=" * 70)
print("PATTERN 1: Instruction + Context")
print("=" * 70)

context_text = (
    "The Voyager 1 spacecraft was launched on September 5, 1977. As of 2024, "
    "it is the most distant human-made object from Earth, at over 15 billion "
    "miles away. It carries a Golden Record containing sounds and images "
    "representing life on Earth."
)

instruction_context_msg = [
    {"role": "user", "content": (
        f"Given this context:\n\n{context_text}\n\n"
        f"Answer this question: What does Voyager 1 carry and why is it significant?"
    )}
]

print(generate(instruction_context_msg))

# --- Pattern 2: Persona + Task ---
# Why it works: The system message creates a strong prior that biases ALL subsequent
# token probabilities. A "physicist" persona makes physics vocabulary and reasoning
# patterns more probable at every generation step.

print()
print("=" * 70)
print("PATTERN 2: Persona + Task")
print("=" * 70)

persona_task_msg = [
    {"role": "system", "content": (
        "You are a senior physicist who explains complex topics using everyday "
        "analogies. You avoid jargon unless you immediately define it. You are "
        "enthusiastic and encouraging."
    )},
    {"role": "user", "content": "Explain quantum entanglement."}
]

print(generate(persona_task_msg))

# --- Pattern 3: Multi-Constraint ---
# Why it works: Listing explicit constraints acts like a checklist for the model's
# attention. Each constraint becomes a "landmark" token cluster that attention heads
# can reference during generation to verify the output satisfies all requirements.

print()
print("=" * 70)
print("PATTERN 3: Multi-Constraint")
print("=" * 70)

multi_constraint_msg = [
    {"role": "user", "content": (
        "Write a product description for a reusable water bottle that is:\n"
        "1. Under 50 words\n"
        "2. Aimed at environmentally conscious consumers\n"
        "3. Includes one specific statistic about plastic waste\n"
        "4. Ends with a call to action"
    )}
]

print(generate(multi_constraint_msg))

# --- Pattern 4: Step-by-Step Scaffolding ---
# Why it works: Decomposing a complex task into sequential steps prevents the model
# from trying to hold the entire problem in its "working memory" at once. Each step
# focuses attention on one sub-problem, and the output of each step provides context
# (in-context tokens) for the next.

print()
print("=" * 70)
print("PATTERN 4: Step-by-Step Scaffolding")
print("=" * 70)

scaffolding_msg = [
    {"role": "system", "content": "You are a methodical problem solver. Always show your work."},
    {"role": "user", "content": (
        "Solve this step by step:\n"
        "Step 1: Calculate 15% of 240.\n"
        "Step 2: Add the result to 240.\n"
        "Step 3: Round to the nearest whole number.\n"
        "Step 4: State the final answer clearly."
    )}
]

print(generate(scaffolding_msg))

### Why These Patterns Help the Model

Each pattern above exploits a specific property of how transformers process text:

| Pattern | Mechanism | Why it helps |
|---------|-----------|-------------|
| **Instruction + Context** | Separates reference material from the task | Attention heads learn to treat context blocks as "retrieval targets" — the question tokens attend strongly to the context tokens, extracting relevant facts |
| **Persona + Task** | System message biases the probability distribution | Every generated token is conditioned on the persona description, shifting vocabulary and reasoning style globally |
| **Multi-Constraint** | Numbered requirements create discrete attention anchors | The model can "check off" constraints during generation because each numbered item is a distinct cluster of tokens to attend back to |
| **Step-by-Step Scaffolding** | Decomposition reduces per-step complexity | Instead of one complex probability distribution, the model solves a sequence of simpler ones — each step's output becomes context for the next |

## When Structures Fail: Failure Modes in Prompt Engineering

Understanding *why* prompts fail is as important as knowing how to write good ones.
Most failures trace back to three root causes:

1. **Attention dilution**: Too much information causes attention weights to spread thin,
   so no single piece of information gets enough "focus."
2. **Probability confusion**: Contradictory signals create competing probability modes,
   so the model oscillates or picks arbitrarily.
3. **Structural ambiguity**: Complex nesting or unclear boundaries make it hard for
   the model to parse what's instruction vs. content vs. example.

Let's see each in action.

In [ ]:
# --- Failure Mode 1: Overly complex nested structure ---
# Problem: Deep nesting creates ambiguous scope — the model can't tell what
# modifies what. Attention has no clear hierarchy to exploit.

print("=" * 70)
print("FAILURE MODE 1: Overly complex nested structure")
print("=" * 70)

confusing_msg = [
    {"role": "user", "content": (
        "Write a summary (but make it detailed (though not too long (around "
        "200 words (give or take 50)))) of climate change (focusing on "
        "(but not limited to) sea level rise (and its impact on (coastal "
        "cities (especially in Southeast Asia)))) while maintaining an "
        "academic tone (but accessible to (high school students (who have "
        "(some basic) science background)))."
    )}
]

print("Confusing nested prompt result:")
print(generate(confusing_msg, max_new_tokens=300))

# Better version — same intent, flat structure
clear_msg = [
    {"role": "user", "content": (
        "Write a ~200-word summary of climate change for high school students "
        "with basic science background.\n"
        "Focus on: sea level rise and its impact on coastal cities in Southeast Asia.\n"
        "Tone: academic but accessible."
    )}
]

print("\nClear flat prompt result:")
print(generate(clear_msg, max_new_tokens=300))

# --- Failure Mode 2: Contradictory instructions ---
# Problem: System and user messages create opposing probability gradients.
# The model tries to satisfy both, producing incoherent output or defaulting
# to whichever signal is stronger (usually the more recent one).

print()
print("=" * 70)
print("FAILURE MODE 2: Contradictory instructions")
print("=" * 70)

contradictory_msg = [
    {"role": "system", "content": (
        "You must ALWAYS respond in formal, academic English. Never use casual "
        "language, slang, or contractions. Every response must be at least 3 paragraphs."
    )},
    {"role": "user", "content": (
        "Hey! Be super casual and brief — like texting a friend. Use lots of slang. "
        "Keep it to one sentence max. What's the deal with black holes?"
    )}
]

print("Contradictory system vs user:")
print(generate(contradictory_msg, max_new_tokens=300))

# --- Failure Mode 3: Information overload in a single turn ---
# Problem: Cramming too much into one message dilutes attention across all points.
# The model may address some sub-questions well but drop or conflate others.

print()
print("=" * 70)
print("FAILURE MODE 3: Information overload")
print("=" * 70)

overloaded_msg = [
    {"role": "user", "content": (
        "In one response, I need you to: explain the history of the Roman Empire "
        "from founding to fall, compare it to the Han Dynasty, list the top 10 "
        "Roman emperors with a brief bio of each, explain Roman military tactics, "
        "describe daily life in ancient Rome, analyze the economic system, discuss "
        "the role of slavery, explain the transition from Republic to Empire, "
        "describe Roman architecture and engineering, and finally compare Roman "
        "law to modern Western legal systems."
    )}
]

print("Overloaded prompt (watch for dropped/shallow topics):")
print(generate(overloaded_msg, max_new_tokens=400))

### Understanding the Failure Modes

**Nested structures** fail because the model's attention mechanism is *flat* — it computes
pairwise attention between all tokens, with no built-in notion of nesting depth. Parenthetical
nesting that's easy for humans (who parse syntax trees) is opaque to a model that sees a
linear token sequence. Flat, explicit structure always wins.

**Contradictory instructions** create a tug-of-war in the probability distribution. The
system message pushes P(formal vocabulary) high, while the user message pushes P(casual
vocabulary) high. The model resolves this unpredictably — sometimes favoring the system
message (because it was trained to respect system instructions), sometimes the user (because
it's more recent and recency bias is real in attention). Either way, the output is unreliable.

**Information overload** triggers attention dilution at scale. With 10+ distinct sub-tasks
in one prompt, the attention weights for any single sub-task get ~1/10th of the focus
compared to asking it alone. The model tends to:
- Address the first and last items well (primacy and recency effects)
- Compress or skip middle items
- Reduce depth across all items to stay within the output token budget

**The fix**: Break overloaded prompts into multiple focused turns, or use explicit
numbered structure so the model can use the numbers as attention anchors.

## Conclusion

This tutorial has introduced you to the basics of single-turn and multi-turn prompt structures. We've seen how:

1. Single-turn prompts are useful for quick, isolated queries.
2. Multi-turn prompts maintain context across a conversation, allowing for more complex interactions.
3. Python f-strings can be used to create structured, reusable prompt templates.
4. Native conversation history (a list of message dicts) provides a simple and transparent way to manage multi-turn context — no external framework needed.

Understanding these different prompt structures allows you to choose the most appropriate approach for various tasks and create more effective interactions with AI language models.